In [20]:
#Ridge Regression

import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split,KFold,cross_val_score,GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
from sklearn.preprocessing import StandardScaler,OrdinalEncoder
import mlflow


## load the dataset preprocess the data

In [12]:
df=pd.read_csv("D:/powerbi/medical_insurance.csv")
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [15]:
x=df[['age', 'sex', 'bmi', 'children', 'smoker']]
y=df["charges"]

# select string columns from x
x_str=x.select_dtypes(exclude="number")

#select number values
x_num=x.select_dtypes(include="number")

## create  ColumnTransformer for preprocessing

In [16]:
preprocess=ColumnTransformer(
    transformers=[
        ("numeric values",StandardScaler(),x_num.columns),
        ("string value",OrdinalEncoder(),x_str.columns)
    ],
    remainder="passthrough"
)

create the pipeline

In [17]:
pipeline=Pipeline(
    steps=[
        ("preprocess",preprocess),
        ("model",Ridge())
    ]
)

In [18]:
param={
    "model__alpha":[0.1,0.3,0.5,0.7,0.9,1]
}

In [19]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [25]:
cross=KFold(n_splits=5,random_state=42,shuffle=True)
cross_valid=(cross_val_score(pipeline,x_train,y_train,cv=cross,scoring="r2"))
print(np.mean(cross_valid))

0.7503935265012212


In [28]:
model=GridSearchCV(
    estimator=pipeline,
    cv=cross,
    param_grid=param,
    scoring="r2",
    
)

In [29]:
model.fit(x_train,y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...l', Ridge())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__alpha': [0.1, 0.3, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displaye

In [32]:
print(model.best_params_)

{'model__alpha': 0.3}


In [33]:
y_pred=model.predict(x_test)
r2=r2_score(y_test,y_pred)
mse=mean_squared_error(y_test,y_pred)
mae=mean_absolute_error(y_test,y_pred)
print(f"r2 score: {r2}  || MSE: {mse}  || MAE: {mae}")

r2 score: 0.7387498013304299  || MSE: 40096930.55424512  || MAE: 4171.272417327435


In [35]:
mlflow.set_tracking_uri("http://127.0.0.1:5000/")
mlflow.set_experiment("Medical_Insurance_cost_prediction")

with mlflow.start_run(run_name="Ridge Regression"):
    mlflow.log_params(model.best_params_)
    mlflow.log_metric("R2 score",r2)
    mlflow.log_metric("MSE",mse)
    mlflow.log_metric("MAE",mae)
    mlflow.sklearn.log_model(model,"Ridge_regression")

2026/03/31 12:04:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/31 12:04:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Ridge Regression at: http://127.0.0.1:5000/#/experiments/1/runs/358352f4a5b24ca8943c163ce5d89884
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
